# Covariance dependents testing
This is an example how to create functionally independent set of covariants (equivariant) version of a graphs contraction.


## 0. Tensor Contraction Formats (Cleaned-Up Version)

A tensor contraction is a product of tensors where shared indices are summed.

### a) Raw Tensor Contraction

For a rank-2 tensor \(T\), a simple scalar contraction is

$$
s = T^{(2)}_{ab} T^{(2)}_{bc} T^{(2)}_{ca}
=
\sum_{a,b,c} T^{(2)}_{ab} T^{(2)}_{bc} T^{(2)}_{ca}. 
$$

This is a scalar, or rank-0 contraction, because every index appears exactly twice and is summed out.

If an index appears only once, it remains as a free index, or dangling edge, and the result is still a tensor. For example,

$$
v_a = T^{(2)}_{ab} T^{(2)}_{bc} T^{(1)}_c
$$

has one free index \(a\), so the output has rank 1.

Note that contractions of symmetric trace-free tensors are not necessarily symmetric trace-free tensors unless the remaining free indices are projected back into the STF basis.

### b) Einsum Format

The same scalar contraction can be written as

```python
torch.einsum("ij,jk,ki->", T2, T2, T2)
torch.einsum("ij,jk,k->i", T2, T2, T1)
```

Repeated labels are summed. Labels after `->` are the output indices. Since there are no output labels in the first one, the result is a scalar and second one is a vector because it has single free rank. 

### c) Multigraph Format

A contraction can also be represented as a multigraph:

- tensors are nodes
- tensor ranks are node degrees
- contracted index pairs are edges
- multiple contractions between the same two tensors are multi-edges
- free indices are dangling edges

For $T^{(2)}_{ab} T^{(2)}_{bc} T^{(2)}_{ca}$ the graph is a triangle:

```text
A -- B
 \  /
  C
```
For $T^{(2)}_{ab} T^{(2)}_{bc} T^{(1)}_{c}$ the graph is a triangle:
```text
o
|
T2 -- T2
      |
      T1
```    

In this project, we usually focus on connected multigraphs. A contraction with one tensor rank is called **homogeneous**. A contraction with two tensor ranks is called **pair-heterogeneous**.

### d) Formula Format

The code uses a compact representation similar to einsum. We call this the **formula format**:

```python
(2, 2, 2), ((0, 2), (0, 1), (1, 2))
```

The first tuple gives the tensor ranks. The second tuple gives the edge labels attached to each tensor:

```text
node 0: rank 2, edges (0, 2)
node 1: rank 2, edges (0, 1)
node 2: rank 2, edges (1, 2)
```

This corresponds to

$$
T^{(2)}_{ac} T^{(2)}_{ab} T^{(2)}_{bc}.
$$

The particular edge labels do not matter; only the pattern of shared labels matters.

The dangling edge contraction would be writen. 

```python
(2, 2, 1), ((0, 2), (0, 1), (1,))
``` 



## Independence testing - Handcrafted basis
Let's start with basis, that we develop by hand. 
### 0. Print Basis

In [ ]:
import sys
sys.path.append('..')

In [ ]:
import matplotlib.pyplot as plt
from notebooks.contraction_formats import graph_from_formula
from notebooks.contraction_visualization import draw_graph_example

from moment_invariant_tools.contraction_formats import is_valid_einsum, einsum_to_formula
covariant_basis = ["i -> i", 
                   "i, ij -> j",
                   "ij, ij",
                   "ij, jk, ik", 
                   ]
fig, axs = plt.subplots(1, 4, figsize=(20, 5))
flat_axs = axs.flatten()

for i, basis in enumerate(covariant_basis):
    assert is_valid_einsum(basis), f"Basis {i} is not valid: {basis}"
    formula = einsum_to_formula(basis) 
    title = f"Basis {i}:\n {basis} \n {formula}"
    graph = graph_from_formula(formula)
    draw_graph_example(ax=flat_axs[i], G=graph, title=title)

### 1. Independence testing
Get the different tensor, randomly initialize them. Tensors are saved as (2l + 1) values and mapped to tensor dimension. (NOTE: Does this save memory?)


In [ ]:
import torch
from moment_invariant_tools.construct_tensors import cartesian_irreducible_mapping, sample_reduced_tensor

# Prepare 
mapping = cartesian_irreducible_mapping
sampler = sample_reduced_tensor

rank_set = []
for basis in covariant_basis:
    ranks = einsum_to_formula(basis)[0]
    rank_set.extend(ranks)
rank_set = list(set(rank_set))

mapper_set = {r: mapping(r).to(torch.float64) for r in rank_set}
tensor_set = {r: sampler(r).requires_grad_(True) for r in rank_set}

### 2. Calculate jacobian
We calculate the contraction for the random tensors and calculate gradients append them as rows to form the Jacobian and calculate its rank. 

In [ ]:
from moment_invariant_tools.search_covariance import perform_covariant_grad, test_covariants

def test_independence(basis, tensor_set, mapper_set):
    grad_matrix = None
    current_dof = 0
    contractions = []
    for basis in covariant_basis:
        term, contraction = einsum_to_formula(basis)
        grads_list = perform_covariant_grad(ranks=term,
                                            contraction=contraction,
                                            tensor_set=tensor_set,
                                            mapper_set=mapper_set)
        new_total, new_rows = test_covariants(grads_list, grad_matrix, rank_set)
        if new_total > current_dof:
            print(f"Found new independent covariant: {basis} with {new_total - current_dof} new dof")
            current_dof = new_total 
            contractions.append((term, contraction))
            grad_matrix = torch.cat([grad_matrix, new_rows], dim=0) if grad_matrix is not None else new_rows
        else:
            print(f"Covariant {basis} is not independent, no new dof found.")

test_independence(covariant_basis, tensor_set, mapper_set)

### 3. Test Flexibility 
The way, how we produce invariants and covariance allow degenerate cases


In [ ]:
# This is vanishing example
tensor_set[1] = torch.zeros_like(tensor_set[1], requires_grad=True, dtype=torch.float64)
tensor_set[2] = torch.tensor([-2.0, 1.0, 1.0, 1.0, -2.0,], requires_grad=True, dtype=torch.float64)

# In certain cases of the input tensors, the covariants can be dependent. This is a vanishing example. Where we'll lose a 3 DOF when rank-1 is zero. 
test_independence(covariant_basis, tensor_set, mapper_set)

### 4. Run covariant search on graphs - Homegenous Covariant (single rank contractions)
For odd ranked nodes, we can form 1-dangling edge graphs that will be covariant and the group acts as a rotation matrix on them 


In [ ]:
from logging import basicConfig, WARNING
from moment_invariant_tools.search_covariance import search_homogenous_covariants, build_cov_grad_matrix
from moment_invariant_tools.search_invariants import setup_search 
from moment_invariant_tools.contraction_formats import is_valid_formula, formula_to_einsum
l_max = 3
n_max = 5 
# Search through all odd ranks up to l_max, and all contractions up to n_max.
rank_set = list(range(1, l_max + 1, 1))
odd_ranks = [r for r in rank_set if r % 2 == 1]
grad_map, mapper_set, tensor_set = setup_search(rank_set, spherical=True)
basicConfig(level=WARNING, force=True)

search_homogenous_covariants(grad_map, odd_ranks, n_max, tensor_set, mapper_set)
# Construct the gradient matrix from the gradient map and compute the total number of independent invariants.
grad_matrix = build_cov_grad_matrix(grad_map, rank_set)
# Calculate expected degrees of freedom based on the rank
dof = [2 * r + 1 for r in rank_set]
total_invariants = torch.linalg.matrix_rank(grad_matrix, atol=None, rtol=None).item()
print(f"Total number of independent invariants found: {total_invariants} from {sum(dof)} expected degrees of freedom for ranks {rank_set} (DOF: {dof}) and n_max={n_max}.")

fig, axs = plt.subplots(1, len(grad_map.keys()), figsize=(20, 5))
flat_axs = axs.flatten()

# Plot the contractions 
for i, formula in enumerate(grad_map.keys()):
    is_valid_formula(formula)
    einsum = formula_to_einsum(formula) 
    title = f"Covariant {i}:\n {einsum}"
    graph = graph_from_formula(formula)
    draw_graph_example(ax=flat_axs[i], G=graph, title=title)

### 5. Run covariant search on graphs - Heteregenous Covariant (two rank contractions with odd and even number) 

In [ ]:
import logging
from moment_invariant_tools.search_covariance import search_heterogeneous_covariants
logging.basicConfig(level=logging.WARNING, force=True)

# TODO: Add skipping of already found covariants.
search_heterogeneous_covariants(grad_map, rank_set, n_max, tensor_set, mapper_set)
# Construct the gradient matrix from the gradient map and compute the total number of independent invariants.
grad_matrix = build_cov_grad_matrix(grad_map, rank_set)
# Calculate expected degrees of freedom based on the rank
dof = [2 * r + 1 for r in rank_set]
total_invariants = torch.linalg.matrix_rank(grad_matrix, atol=None, rtol=None).item()


mixed_contractions = [f for f in grad_map.keys() if len(set(f[0])) != 1]

fig, axs = plt.subplots(1, len(mixed_contractions), figsize=(20, 5))
flat_axs = axs.flatten()

for i, formula in enumerate(mixed_contractions):
    is_valid_formula(formula)
    einsum = formula_to_einsum(formula) 
    title = f"Covariant {i}:\n {einsum}"
    graph = graph_from_formula(formula)
    draw_graph_example(ax=flat_axs[i], G=graph, title=title)